#  Notebook 01 — Batch Ingestion & Data Sources
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 1 — Data Ingestion & Sources

---

**Objetivo:** Coletar dados históricos do Citi Bike (viagens), dados climáticos de NYC e feriados dos EUA, armazenando tudo na camada **Bronze (Raw Landing)** do Data Lake em formato Parquet.

**Fontes de dados:**
| Fonte | Tipo | Formato | Método |
|-------|------|---------|--------|
| Citi Bike Trip Data | Estruturado | CSV (zip) | Download S3 Bucket |
| Open-Meteo Historical API | Estruturado | JSON → CSV | REST API |
| Feriados EUA/NY | Estruturado | Gerado | Biblioteca `holidays` |
| GBFS Station Info | Semi-estruturado | JSON | REST API |

**Ferramentas de Big Data utilizadas:**
- **PySpark** — Processamento distribuído dos CSVs massivos
- **PyArrow + Parquet** — Formato colunar otimizado para armazenamento
- **requests** — Ingestão via API REST

**Arquitetura de pastas (Data Lake local):**
```
dados/
├── bronze/          # Raw Landing — dados brutos como foram coletados
│   ├── trips/       # Parquet particionado por ano/mês
│   ├── weather/     # Parquet com clima horário de NYC
│   ├── holidays/    # Parquet com feriados
│   └── stations/    # JSON das estações GBFS
├── silver/          # Trusted/Cleaned (notebook 03)
└── gold/            # Business/Refined (notebook 04)
```

## 0. Setup & Dependências

In [ ]:
!pip install pyspark pyarrow pandas requests holidays openmeteo-requests requests-cache retry-requests tqdm -q

In [2]:
import os
import sys
import json
import zipfile
import glob
import logging
from pathlib import Path
from datetime import datetime, timedelta
from io import BytesIO

import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import holidays
from tqdm import tqdm

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, IntegerType, FloatType
)

# Configuração de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_ingestion')

print(f"Python {sys.version}")
print(f"Pandas {pd.__version__}")
print(f"PyArrow {pa.__version__}")

Python 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Pandas 2.0.3
PyArrow 13.0.0


In [ ]:
# ============================================================
# CONFIGURACAO
# ============================================================

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / 'dados'

# Camadas do Data Lake
BRONZE_DIR      = DATA_DIR / 'bronze'
BRONZE_TRIPS    = BRONZE_DIR / 'trips'
BRONZE_WEATHER  = BRONZE_DIR / 'weather'
BRONZE_HOLIDAYS = BRONZE_DIR / 'holidays'
BRONZE_STATIONS = BRONZE_DIR / 'stations'

# Diretorio temporario para downloads
RAW_DOWNLOADS = DATA_DIR / 'raw_downloads'

# Criar todas as pastas
for d in [BRONZE_TRIPS, BRONZE_WEATHER, BRONZE_HOLIDAYS, BRONZE_STATIONS, RAW_DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Periodo de ingestao
# O Citi Bike publica dados com ~2 meses de atraso.
# END_YEAR/END_MONTH sao calculados automaticamente subtraindo 2 meses da data atual.
# Para testes, fixamos START_YEAR/START_MONTH em um valor mais recente, dado o grande fluxo de dados disponíveis. Não recomendamos setar para o ano inicail disponível (2013), pois gera aproximadamente 250gb em dados CSV e 45gb em dados Parquet.
START_YEAR  = 2025
START_MONTH = 8

_today      = datetime.now()
_lag        = 2  # meses de atraso tipico da publicacao
_end        = _today.replace(day=1)
for _ in range(_lag):
    _end = (_end - timedelta(days=1)).replace(day=1)
END_YEAR  = _end.year
END_MONTH = _end.month

# Coordenadas de NYC (Central Park) para dados climaticos
NYC_LAT = 40.7831
NYC_LON = -73.9712

total_months = (END_YEAR - START_YEAR) * 12 + END_MONTH - START_MONTH + 1
print(f'Raiz do projeto: {PROJECT_ROOT}')
print(f'Data Lake: {DATA_DIR}')
print(f'Periodo: {START_YEAR}-{START_MONTH:02d} ate {END_YEAR}-{END_MONTH:02d}')
print(f'Total de meses estimado: {total_months}')


Raiz do projeto: /home/jovyan/work
Data Lake: /home/jovyan/work/dados
Periodo: 2023-01 ate 2026-02
Total de meses estimado: 38


In [4]:
# ============================================================
# Inicializar SparkSession
# ============================================================

import sys
import platform

# Configurar o executável Python para o Spark (no Windows não existe "python3")
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Garantir que o bin do HADOOP_HOME está no PATH (HADOOP_HOME já vem do ambiente do sistema)
if platform.system() == 'Windows':
    _hadoop_home = os.environ.get('HADOOP_HOME', '')
    if _hadoop_home:
        _hadoop_bin = os.path.join(_hadoop_home, 'bin')
        if _hadoop_bin not in os.environ.get('PATH', ''):
            os.environ['PATH'] = _hadoop_bin + ';' + os.environ.get('PATH', '')

# Parar sessão anterior para que as novas configs sejam aplicadas
from pyspark.sql import SparkSession as _SS
_active = _SS.getActiveSession()
if _active is not None:
    _active.stop()

spark = (
    SparkSession.builder
    .appName('CitiBike_BatchIngestion')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.warehouse.dir', str(DATA_DIR / 'spark-warehouse').replace('\\', '/'))
    .config('spark.local.dir', str(DATA_DIR / 'spark-temp').replace('\\', '/'))
    .master('local[*]')
    .getOrCreate()
)

# Injetar hadoop.home.dir no JVM — necessário no Windows para o Spark writer nativo
if platform.system() == 'Windows':
    _hadoop_home = os.environ.get('HADOOP_HOME', '')
    if _hadoop_home:
        spark._jvm.System.setProperty('hadoop.home.dir', _hadoop_home.replace('\\', '/'))
        logger.info(f'hadoop.home.dir configurado no JVM: {_hadoop_home}')

spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} inicializado")
print(f"   Cores: {spark.sparkContext.defaultParallelism}")
print(f"   Driver memory: {spark.conf.get('spark.driver.memory')}")

Spark 3.5.0 inicializado
   Cores: 24
   Driver memory: 4g


---
## 1.  Ingestão Batch — Citi Bike Trip Data

Os dados de viagem do Citi Bike ficam em um bucket público S3 da AWS:  
`https://s3.amazonaws.com/tripdata/`

Cada mês tem um ZIP contendo um ou mais CSVs. O schema mudou ao longo dos anos:
- **Schema antigo** (até ~2021): `tripduration`, `starttime`, `stoptime`, `start station id`, etc.
- **Schema novo** (2021+): `ride_id`, `rideable_type`, `started_at`, `ended_at`, `start_station_name`, etc.

Vamos baixar, descompactar e converter para Parquet particionado usando **PySpark**.

In [5]:
# ============================================================
# 1.1 Gerar lista de URLs dos ZIPs do Citi Bike
# ============================================================

def generate_citibike_urls(start_year, start_month, end_year, end_month):
    """
    Gera todas as URLs possíveis para os ZIPs do Citi Bike.

    Regras de nomenclatura confirmadas no bucket S3:
      - Anos <= 2023 : ZIP ÚNICO por ano → {ano}-citibike-tripdata.zip
                       (ex: 2023-citibike-tripdata.zip)
      - 2024 Jan-Abr : mensal, .csv.zip primeiro
                       (ex: 202401-citibike-tripdata.csv.zip)
      - 2024 Mai+    : mensal, .zip primeiro
                       (ex: 202405-citibike-tripdata.zip)
    """
    base_url = 'https://s3.amazonaws.com/tripdata'
    urls = []
    years_seen = set()

    current = datetime(start_year, start_month, 1)
    end     = datetime(end_year, end_month, 1)

    while current <= end:
        year  = current.year
        month = current.month

        if year <= 2023:
            # Um único ZIP cobre o ano inteiro — emitir só uma vez por ano
            if year not in years_seen:
                years_seen.add(year)
                urls.append({
                    'year'      : year,
                    'month'     : None,   # anual — mês será inferido do CSV
                    'annual'    : True,
                    'candidates': [f'{base_url}/{year}-citibike-tripdata.zip'],
                    'label'     : str(year),
                })
        else:
            ym = current.strftime('%Y%m')  # ex: 202401

            # Jan-Abr 2024: padrão .csv.zip veio primeiro historicamente
            if month <= 4:
                candidates = [
                    f'{base_url}/{ym}-citibike-tripdata.csv.zip',
                    f'{base_url}/{ym}-citibike-tripdata.zip',
                ]
            else:
                candidates = [
                    f'{base_url}/{ym}-citibike-tripdata.zip',
                    f'{base_url}/{ym}-citibike-tripdata.csv.zip',
                ]

            urls.append({
                'year'      : year,
                'month'     : month,
                'annual'    : False,
                'candidates': candidates,
                'label'     : current.strftime('%Y-%m'),
            })

        # Avançar para o próximo mês
        if current.month == 12:
            current = datetime(current.year + 1, 1, 1)
        else:
            current = datetime(current.year, current.month + 1, 1)

    return urls

url_list = generate_citibike_urls(START_YEAR, START_MONTH, END_YEAR, END_MONTH)
print(f"Total de entradas para download: {len(url_list)}")
for u in url_list[:3]:
    print(f"   {u['label']}: {u['candidates'][0]}")
print(f"   ...")
for u in url_list[-2:]:
    print(f"   {u['label']}: {u['candidates'][0]}")


Total de entradas para download: 27
   2023: https://s3.amazonaws.com/tripdata/2023-citibike-tripdata.zip
   2024-01: https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.csv.zip
   2024-02: https://s3.amazonaws.com/tripdata/202402-citibike-tripdata.csv.zip
   ...
   2026-01: https://s3.amazonaws.com/tripdata/202601-citibike-tripdata.csv.zip
   2026-02: https://s3.amazonaws.com/tripdata/202602-citibike-tripdata.csv.zip


In [6]:
# ============================================================
# 1.2 Download dos ZIPs com retry e verificação
# ============================================================

def download_citibike_zip(url_info, output_dir, max_retries=3):
    """
    Baixa o ZIP de um mês do Citi Bike, tentando múltiplos padrões de URL.
    Retorna o caminho do ZIP baixado ou None se falhar.
    """
    label = url_info['label']
    
    for url in url_info['candidates']:
        filename = url.split('/')[-1]
        output_path = output_dir / filename
        
        # Pular se já foi baixado
        if output_path.exists() and output_path.stat().st_size > 1000:
            logger.info(f"⏭  {label}: já existe ({output_path.stat().st_size / 1e6:.1f} MB)")
            return output_path
        
        for attempt in range(max_retries):
            try:
                # Verificar se a URL existe (HEAD request)
                head = requests.head(url, timeout=10)
                if head.status_code != 200:
                    break  # tentar próximo padrão
                
                file_size = int(head.headers.get('content-length', 0))
                logger.info(f"⬇  {label}: baixando {filename} ({file_size/1e6:.1f} MB)...")
                
                # Download com streaming
                response = requests.get(url, stream=True, timeout=300)
                response.raise_for_status()
                
                with open(output_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192 * 16):
                        f.write(chunk)
                
                logger.info(f" {label}: download completo ({output_path.stat().st_size / 1e6:.1f} MB)")
                return output_path
                
            except requests.RequestException as e:
                logger.warning(f"⚠️  {label}: tentativa {attempt+1}/{max_retries} falhou — {e}")
                if output_path.exists():
                    output_path.unlink()
    
    logger.error(f" {label}: nenhuma URL funcionou")
    return None

# Executar downloads
downloaded_zips = []
for url_info in tqdm(url_list, desc='Download Citi Bike'):
    result = download_citibike_zip(url_info, RAW_DOWNLOADS)
    if result:
        downloaded_zips.append({
            'path': result,
            'year': url_info['year'],
            'month': url_info['month'],
            'label': url_info['label']
        })

print(f"\n Total de ZIPs baixados: {len(downloaded_zips)}/{len(url_list)}")

Download Citi Bike:   0%|          | 0/27 [00:00<?, ?it/s]22:08:59 [INFO] ⏭  2023: já existe (1602.4 MB)
22:08:59 [INFO] ⏭  2024-01: já existe (369.0 MB)
Download Citi Bike:  19%|█▊        | 5/27 [00:01<00:06,  3.16it/s]22:09:00 [INFO] ⏭  2024-05: já existe (809.1 MB)
22:09:00 [INFO] ⏭  2024-06: já existe (936.0 MB)
22:09:00 [INFO] ⏭  2024-07: já existe (923.3 MB)
22:09:00 [INFO] ⏭  2024-08: já existe (900.0 MB)
22:09:00 [INFO] ⏭  2024-09: já existe (976.5 MB)
22:09:00 [INFO] ⏭  2024-10: já existe (1006.6 MB)
22:09:00 [INFO] ⏭  2024-11: já existe (724.3 MB)
22:09:00 [INFO] ⏭  2024-12: já existe (450.8 MB)
22:09:00 [INFO] ⏭  2025-01: já existe (414.2 MB)
Download Citi Bike:  52%|█████▏    | 14/27 [00:01<00:01, 10.80it/s]22:09:01 [INFO] ⏭  2025-02: já existe (396.0 MB)
22:09:01 [INFO] ⏭  2025-03: já existe (618.3 MB)
Download Citi Bike:  63%|██████▎   | 17/27 [00:03<00:01,  5.24it/s]22:09:02 [INFO] ⏭  2025-05: já existe (844.4 MB)
22:09:02 [INFO] ⏭  2025-06: já existe (929.0 MB)
22:09:02


 Total de ZIPs baixados: 27/27


In [ ]:
# ============================================================
# 1.3 Extrair CSVs dos ZIPs
# ============================================================

import re as _re

def _infer_year_month(filename):
    """Extrai (ano, mês) do nome de arquivo como 202301-citibike-tripdata.csv."""
    m = _re.match(r'(\d{4})(\d{2})-', Path(filename).name)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None

def extract_csvs_from_zip(zip_path, output_dir):
    """
    Extrai todos os CSVs de dentro do ZIP.
    Retorna lista de caminhos dos CSVs extraídos.
    Suporta ZIPs anuais (ex: 2023) que contêm CSVs de vários meses.
    """
    csv_paths = []
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            csv_files = [f for f in z.namelist()
                         if f.lower().endswith('.csv') and not f.startswith('__MACOSX')]

            for csv_file in csv_files:
                out_path = output_dir / Path(csv_file).name
                if not out_path.exists():
                    z.extract(csv_file, output_dir)
                    # Mover para o diretório raiz se estava em subpasta
                    extracted = output_dir / csv_file
                    if extracted != out_path:
                        extracted.rename(out_path)
                csv_paths.append(out_path)

    except zipfile.BadZipFile:
        logger.error(f" ZIP corrompido: {zip_path}")

    return csv_paths

# Extrair todos
all_csv_paths = []
for z_info in tqdm(downloaded_zips, desc='Extraindo CSVs'):
    csvs = extract_csvs_from_zip(z_info['path'], RAW_DOWNLOADS)
    for csv_path in csvs:
        if z_info.get('annual'):
            # ZIP anual: inferir ano/mês do nome do CSV interno
            year, month = _infer_year_month(csv_path.name)
            if year is None:
                year  = z_info['year']
                month = None
                logger.warning(f"  Não foi possível inferir mês de '{csv_path.name}' — year={year}")
        else:
            year  = z_info['year']
            month = z_info['month']

        all_csv_paths.append({
            'path' : str(csv_path),
            'year' : year,
            'month': month,
        })

print(f"\n📄 Total de CSVs extraídos: {len(all_csv_paths)}")
for c in all_csv_paths[:3]:
    print(f"   {c['path']}  (year={c['year']}, month={c['month']})")


In [7]:
# Montar all_csv_paths direto dos CSVs já extraídos
import re as _re

all_csv_paths = []
for csv_path in sorted(RAW_DOWNLOADS.glob('*.csv')):
    m = _re.match(r'(\d{4})(\d{2})-', csv_path.name)
    if m:
        year  = int(m.group(1))
        month = int(m.group(2))
    else:
        year, month = None, None
    all_csv_paths.append({'path': str(csv_path), 'year': year, 'month': month})

print(f"Total de CSVs: {len(all_csv_paths)}")
for c in all_csv_paths[:3]:
    print(f"  {c['path']}  (year={c['year']}, month={c['month']})")

Total de CSVs: 107
  /home/jovyan/work/dados/raw_downloads/202401-citibike-tripdata_1.csv  (year=2024, month=1)
  /home/jovyan/work/dados/raw_downloads/202401-citibike-tripdata_2.csv  (year=2024, month=1)
  /home/jovyan/work/dados/raw_downloads/202402-citibike-tripdata_1.csv  (year=2024, month=2)


In [8]:
# ============================================================
# 1.4 Inspecionar schema dos CSVs (verificação de qualidade)
# ============================================================

# Ler amostra do primeiro CSV para entender o schema
if all_csv_paths:
    sample_df = pd.read_csv(all_csv_paths[0]['path'], nrows=5)
    print("🔍 Colunas do schema (amostra do primeiro CSV):")
    print(f"   {list(sample_df.columns)}")
    print(f"\n🔍 Tipos:")
    print(sample_df.dtypes)
    print(f"\n🔍 Preview:")
    display(sample_df.head(3))
else:
    print("  Nenhum CSV disponível — verifique os downloads acima")

🔍 Colunas do schema (amostra do primeiro CSV):
   ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']

🔍 Tipos:
ride_id                object
rideable_type          object
started_at             object
ended_at               object
start_station_name     object
start_station_id      float64
end_station_name       object
end_station_id        float64
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual          object
dtype: object

🔍 Preview:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,8E865410DBDE0CA9,electric_bike,2024-01-01 13:00:04.563,2024-01-01 13:04:04.652,3 St & 3 Ave,4028.03,Carroll St & Smith St,4225.14,40.675070,-73.987752,40.680611,-73.994758,casual
1,0403D0B3FC9CA77D,electric_bike,2024-01-08 19:36:43.520,2024-01-08 19:53:16.266,Franklin Ave & St Marks Ave,4107.05,Bedford Ave & Bergen St,4066.15,40.675832,-73.956168,40.676368,-73.952918,casual
2,F6DE7BB42FF550BE,electric_bike,2024-01-12 15:00:41.580,2024-01-12 15:36:29.622,W 67 St & Broadway,7116.04,Central Park W & W 103 St,7577.27,40.774925,-73.982666,40.795590,-73.961884,casual


In [9]:
# ============================================================
# 1.5 Carregar CSVs no PySpark e gravar como Parquet (Bronze)
# ============================================================

# Schema esperado do Citi Bike (2023+)
CITIBIKE_SCHEMA = StructType([
    StructField('ride_id', StringType(), True),
    StructField('rideable_type', StringType(), True),
    StructField('started_at', StringType(), True),
    StructField('ended_at', StringType(), True),
    StructField('start_station_name', StringType(), True),
    StructField('start_station_id', StringType(), True),
    StructField('end_station_name', StringType(), True),
    StructField('end_station_id', StringType(), True),
    StructField('start_lat', DoubleType(), True),
    StructField('start_lng', DoubleType(), True),
    StructField('end_lat', DoubleType(), True),
    StructField('end_lng', DoubleType(), True),
    StructField('member_casual', StringType(), True),
])

def ingest_trips_to_bronze(csv_list, output_path, spark_session):
    """
    Lê todos os CSVs de viagem com PySpark, processa, e grava
    como Parquet particionado usando o writer nativo do Spark.
    """
    csv_paths = [item['path'] for item in csv_list]

    logger.info(f"📖 Lendo {len(csv_paths)} CSVs com PySpark...")

    df = (
        spark_session.read
        .option('header', 'true')
        .option('inferSchema', 'false')
        .schema(CITIBIKE_SCHEMA)
        .csv(csv_paths)
    )

    # Converter timestamps e adicionar colunas de partição
    df = (
        df
        .withColumn('started_at', F.to_timestamp('started_at'))
        .withColumn('ended_at', F.to_timestamp('ended_at'))
        .withColumn('year', F.year('started_at'))
        .withColumn('month', F.month('started_at'))
        .withColumn(
            'trip_duration_sec',
            F.unix_timestamp('ended_at') - F.unix_timestamp('started_at')
        )
        .withColumn('ingestion_timestamp', F.current_timestamp())
    )

    # Gravar Parquet particionado com o writer nativo do Spark
    logger.info(f"💾 Gravando Parquet particionado com Spark writer nativo...")
    df.write.partitionBy('year', 'month').mode('overwrite').parquet(str(output_path))

    # Validação rápida lendo o Parquet recém-gravado
    validation = spark_session.read.parquet(str(output_path))
    total = validation.count()
    logger.info(f"✅ {total:,} registros gravados na Bronze")

    return df

# Executar ingestão
if all_csv_paths:
    trips_df = ingest_trips_to_bronze(all_csv_paths, BRONZE_TRIPS, spark)
else:
    print("⚠️  Pule esta célula se não há CSVs baixados (modo offline)")

22:09:02 [INFO] 📖 Lendo 107 CSVs com PySpark...
22:09:04 [INFO] 💾 Gravando Parquet particionado com Spark writer nativo...
22:11:53 [INFO] ✅ 93,110,146 registros gravados na Bronze


In [10]:
# ============================================================
# 1.6 Validação da ingestão de trips
# ============================================================

# Ler os Parquets recém-gravados para validar
if BRONZE_TRIPS.exists() and any(BRONZE_TRIPS.rglob('*.parquet')):
    trips_bronze = spark.read.parquet(str(BRONZE_TRIPS))

    print("=" * 60)
    print("📊 VALIDAÇÃO — Camada Bronze: Trips")
    print("=" * 60)
    print(f"Total de registros: {trips_bronze.count():,}")
    print(f"Total de colunas:   {len(trips_bronze.columns)}")
    print(f"\nSchema:")
    trips_bronze.printSchema()

    print(f"\n📅 Distribuição por ano/mês:")
    (
        trips_bronze
        .groupBy('year', 'month')
        .count()
        .orderBy('year', 'month')
        .show(50, truncate=False)
    )

    print(f"\n🚲 Tipos de bicicleta:")
    trips_bronze.groupBy('rideable_type').count().show()

    print(f"\n👤 Tipos de membro:")
    trips_bronze.groupBy('member_casual').count().show()

    # Verificar nulos
    print(f"\n⚠️  Contagem de nulos por coluna:")
    null_counts = trips_bronze.select(
        [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in trips_bronze.columns]
    )
    null_counts.show(truncate=False)

    # Tamanho em disco
    parquet_size = sum(f.stat().st_size for f in BRONZE_TRIPS.rglob('*.parquet'))
    print(f"\n💾 Tamanho em disco (Parquet): {parquet_size / 1e9:.2f} GB")
else:
    print("⚠️  Nenhum Parquet encontrado em Bronze/trips")

📊 VALIDAÇÃO — Camada Bronze: Trips
Total de registros: 93,110,146
Total de colunas:   17

Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)


📅 Distribuição por ano/mês:
+----+-----+-------+
|year|month|count  |
+----+-----+-------+
|2023|12   |410    |
|2024|1    |1887908|
|2024|2    |21215

---
## 2. 🌤️ Ingestão — Dados Climáticos (Open-Meteo API)

Usamos a API **Open-Meteo Historical Weather** que é:
- ✅ Gratuita (sem API key)
- ✅ Dados horários desde 1940
- ✅ Baseada no ERA5 do ECMWF (referência mundial)

**Variáveis coletadas** (impacto direto no uso de bikes):
- `temperature_2m` — Temperatura (°C)
- `relative_humidity_2m` — Umidade relativa (%)
- `precipitation` — Precipitação (mm)
- `rain` — Chuva (mm)
- `snowfall` — Neve (cm)
- `wind_speed_10m` — Velocidade do vento (km/h)
- `weather_code` — Código WMO de condição climática

In [11]:
# ============================================================
# 2.1 Coletar dados climáticos horários de NYC
# ============================================================

def fetch_weather_openmeteo(lat, lon, start_date, end_date):
    """
    Coleta dados climáticos horários da Open-Meteo Historical API.
    A API aceita no máximo ~2 anos por requisição, então dividimos.
    
    Returns: pd.DataFrame com dados horários
    """
    base_url = 'https://archive-api.open-meteo.com/v1/archive'
    
    hourly_vars = [
        'temperature_2m',
        'relative_humidity_2m',
        'apparent_temperature',
        'precipitation',
        'rain',
        'snowfall',
        'snow_depth',
        'weather_code',
        'wind_speed_10m',
        'wind_gusts_10m',
        'cloud_cover',
    ]
    
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': ','.join(hourly_vars),
        'timezone': 'America/New_York',
        'temperature_unit': 'celsius',
        'wind_speed_unit': 'kmh',
        'precipitation_unit': 'mm',
    }
    
    logger.info(f"🌤️  Buscando clima de {start_date} a {end_date}...")
    
    response = requests.get(base_url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    # Converter para DataFrame
    hourly_data = data['hourly']
    df = pd.DataFrame({
        'datetime': pd.to_datetime(hourly_data['time']),
        **{var: hourly_data[var] for var in hourly_vars}
    })
    
    logger.info(f"   ✅ {len(df):,} registros horários coletados")
    return df

# Coletar clima para todo o período
# A API aceita períodos longos, mas vamos dividir por ano para segurança
weather_dfs = []

for year in range(START_YEAR, END_YEAR + 1):
    s_month = START_MONTH if year == START_YEAR else 1
    e_month = END_MONTH if year == END_YEAR else 12
    
    start_str = f"{year}-{s_month:02d}-01"
    
    # Último dia do mês
    if e_month == 12:
        end_str = f"{year}-12-31"
    else:
        from calendar import monthrange
        last_day = monthrange(year, e_month)[1]
        end_str = f"{year}-{e_month:02d}-{last_day}"
    
    try:
        df_weather = fetch_weather_openmeteo(NYC_LAT, NYC_LON, start_str, end_str)
        weather_dfs.append(df_weather)
    except Exception as e:
        logger.error(f"❌ Erro ao buscar clima para {year}: {e}")

if weather_dfs:
    weather_full = pd.concat(weather_dfs, ignore_index=True)
    print(f"\n📊 Total de registros climáticos: {len(weather_full):,}")
    print(f"   Período: {weather_full['datetime'].min()} → {weather_full['datetime'].max()}")
    display(weather_full.head())

22:12:27 [INFO] 🌤️  Buscando clima de 2023-01-01 a 2023-12-31...
22:12:29 [INFO]    ✅ 8,760 registros horários coletados
22:12:29 [INFO] 🌤️  Buscando clima de 2024-01-01 a 2024-12-31...
22:12:30 [INFO]    ✅ 8,784 registros horários coletados
22:12:30 [INFO] 🌤️  Buscando clima de 2025-01-01 a 2025-12-31...
22:12:31 [INFO]    ✅ 8,760 registros horários coletados
22:12:31 [INFO] 🌤️  Buscando clima de 2026-01-01 a 2026-02-28...
22:12:32 [INFO]    ✅ 1,416 registros horários coletados



📊 Total de registros climáticos: 27,720
   Período: 2023-01-01 00:00:00 → 2026-02-28 23:00:00


,datetime,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,weather_code,wind_speed_10m,wind_gusts_10m,cloud_cover
0,2023-01-01 00:00:00,10.9,99,9.5,1.1,1.1,0.0,0.0,55,11.7,19.1,100
1,2023-01-01 01:00:00,10.9,98,9.7,0.9,0.9,0.0,0.0,53,9.1,22.7,100
2,2023-01-01 02:00:00,10.8,96,8.8,0.1,0.1,0.0,0.0,51,14.4,23.0,94
3,2023-01-01 03:00:00,10.8,95,8.4,0.0,0.0,0.0,0.0,0,16.6,27.4,9
4,2023-01-01 04:00:00,10.1,94,7.4,0.0,0.0,0.0,0.0,0,16.9,28.4,1


In [12]:
# ============================================================
# 2.2 Gravar dados climáticos como Parquet (Bronze)
# ============================================================

if weather_dfs:
    # Adicionar colunas de partição
    weather_full['year'] = weather_full['datetime'].dt.year
    weather_full['month'] = weather_full['datetime'].dt.month
    weather_full['date'] = weather_full['datetime'].dt.date.astype(str)
    weather_full['hour'] = weather_full['datetime'].dt.hour

    # Classificação simplificada do tempo para features
    def classify_weather(code):
        """Classifica o código WMO em categorias simples."""
        if pd.isna(code):
            return 'unknown'
        code = int(code)
        if code <= 3:
            return 'clear'
        elif code <= 49:
            return 'fog'
        elif code <= 69:
            return 'rain'
        elif code <= 79:
            return 'snow'
        elif code <= 99:
            return 'storm'
        return 'other'

    weather_full['weather_category'] = weather_full['weather_code'].apply(classify_weather)
    weather_full['ingestion_timestamp'] = datetime.now()

    # Gravar Parquet com PyArrow (evita dependência do Hadoop no Windows)
    BRONZE_WEATHER.mkdir(parents=True, exist_ok=True)
    _table = pa.Table.from_pandas(weather_full)
    pq.write_table(_table, BRONZE_WEATHER / 'data.parquet', compression='snappy')
    del _table

    # Verificação
    parquet_size = sum(f.stat().st_size for f in BRONZE_WEATHER.rglob('*.parquet'))
    print(f"✅ Clima gravado na Bronze: {parquet_size / 1e6:.1f} MB")
    print(f"   Variáveis: {[c for c in weather_full.columns if c not in ['year','month','ingestion_timestamp']]}")

    # Estatísticas rápidas
    print(f"\n📊 Estatísticas climáticas NYC:")
    display(weather_full[['temperature_2m','precipitation','wind_speed_10m','snow_depth']].describe())


✅ Clima gravado na Bronze: 0.5 MB
   Variáveis: ['datetime', 'temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'wind_speed_10m', 'wind_gusts_10m', 'cloud_cover', 'date', 'hour', 'weather_category']

📊 Estatísticas climáticas NYC:


,temperature_2m,precipitation,wind_speed_10m,snow_depth
count,27720.000000,27720.000000,27720.000000,27720.000000
mean,11.971988,0.146263,11.530152,0.009306
std,10.177063,0.743716,6.103549,0.037313
min,-17.600000,0.000000,0.000000,0.000000
25%,3.800000,0.000000,7.000000,0.000000
50%,12.100000,0.000000,10.600000,0.000000
75%,20.400000,0.000000,15.100000,0.000000
max,39.300000,18.800000,55.000000,0.500000


---
## 3. 📅 Ingestão — Feriados (EUA + NY)

Feriados têm impacto direto na demanda por bikes. Usamos a biblioteca Python `holidays` para gerar todos os feriados federais e estaduais de NY.

In [13]:
# ============================================================
# 3.1 Gerar dataset de feriados
# ============================================================

def generate_holidays_dataset(start_year, end_year):
    """
    Gera um DataFrame com todos os feriados federais dos EUA
    e estaduais de NY para o período especificado.
    """
    # Feriados federais
    us_holidays = holidays.US(years=range(start_year, end_year + 1))
    # Feriados estaduais de NY
    ny_holidays = holidays.US(state='NY', years=range(start_year, end_year + 1))
    
    records = []
    
    # Combinar feriados federais
    for date, name in sorted(us_holidays.items()):
        records.append({
            'date': date,
            'holiday_name': name,
            'holiday_type': 'federal',
            'is_federal': True,
            'is_ny_state': date in ny_holidays,
        })
    
    # Adicionar feriados apenas estaduais (não federais)
    for date, name in sorted(ny_holidays.items()):
        if date not in us_holidays:
            records.append({
                'date': date,
                'holiday_name': name,
                'holiday_type': 'state_ny',
                'is_federal': False,
                'is_ny_state': True,
            })
    
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.day_name()
    df['is_weekend'] = df['date'].dt.dayofweek >= 5
    df['ingestion_timestamp'] = datetime.now()
    
    return df

holidays_df = generate_holidays_dataset(START_YEAR, END_YEAR)
print(f" Total de feriados gerados: {len(holidays_df)}")
print(f"   Período: {holidays_df['date'].min().date()} → {holidays_df['date'].max().date()}")
print(f"   Federais: {holidays_df['is_federal'].sum()}")
print(f"   Estaduais NY: {holidays_df['is_ny_state'].sum()}")
display(holidays_df.head(15))

 Total de feriados gerados: 60
   Período: 2023-01-01 → 2026-12-25
   Federais: 47
   Estaduais NY: 60


,date,holiday_name,holiday_type,is_federal,is_ny_state,year,month,day_of_week,is_weekend,ingestion_timestamp
0,2023-01-01,New Year's Day,federal,True,True,2023,1,Sunday,True,2026-04-07 22:12:32.355386
1,2023-01-02,New Year's Day (observed),federal,True,True,2023,1,Monday,False,2026-04-07 22:12:32.355386
2,2023-01-16,Martin Luther King Jr. Day,federal,True,True,2023,1,Monday,False,2026-04-07 22:12:32.355386
3,2023-02-20,Washington's Birthday,federal,True,True,2023,2,Monday,False,2026-04-07 22:12:32.355386
4,2023-05-29,Memorial Day,federal,True,True,2023,5,Monday,False,2026-04-07 22:12:32.355386
5,2023-06-19,Juneteenth National Independence Day,federal,True,True,2023,6,Monday,False,2026-04-07 22:12:32.355386
6,2023-07-04,Independence Day,federal,True,True,2023,7,Tuesday,False,2026-04-07 22:12:32.355386
7,2023-09-04,Labor Day,federal,True,True,2023,9,Monday,False,2026-04-07 22:12:32.355386
8,2023-10-09,Columbus Day,federal,True,True,2023,10,Monday,False,2026-04-07 22:12:32.355386
9,2023-11-10,Veterans Day (observed),federal,True,True,2023,11,Friday,False,2026-04-07 22:12:32.355386


In [14]:
# ============================================================
# 3.2 Gravar feriados como Parquet (Bronze)
# ============================================================

# Gravar Parquet com PyArrow (evita dependência do Hadoop no Windows)
BRONZE_HOLIDAYS.mkdir(parents=True, exist_ok=True)
_table = pa.Table.from_pandas(holidays_df)
pq.write_table(_table, BRONZE_HOLIDAYS / 'data.parquet', compression='snappy')
del _table

print(f"✅ Feriados gravados na Bronze")

# Ler de volta para validar
_val = pq.read_table(str(BRONZE_HOLIDAYS)).to_pandas()
print(_val.head(10).to_string())
del _val


✅ Feriados gravados na Bronze
        date                          holiday_name holiday_type  is_federal  is_ny_state  year  month day_of_week  is_weekend        ingestion_timestamp
0 2023-01-01                        New Year's Day      federal        True         True  2023      1      Sunday        True 2026-04-07 22:12:32.355386
1 2023-01-02             New Year's Day (observed)      federal        True         True  2023      1      Monday       False 2026-04-07 22:12:32.355386
2 2023-01-16            Martin Luther King Jr. Day      federal        True         True  2023      1      Monday       False 2026-04-07 22:12:32.355386
3 2023-02-20                 Washington's Birthday      federal        True         True  2023      2      Monday       False 2026-04-07 22:12:32.355386
4 2023-05-29                          Memorial Day      federal        True         True  2023      5      Monday       False 2026-04-07 22:12:32.355386
5 2023-06-19  Juneteenth National Independence Day  

---
## 4. 🏢 Ingestão — Estações GBFS (Dados de Referência)

O feed **GBFS (General Bikeshare Feed Specification)** fornece dados em tempo real das estações. Vamos capturar:
- **Station Information** — Dados cadastrais (nome, capacidade, localização)
- **Station Status** — Snapshot atual (bikes disponíveis, docks livres)

In [15]:
# ============================================================
# 4.1 Coletar dados de estações via GBFS
# ============================================================

GBFS_BASE = 'https://gbfs.citibikenyc.com/gbfs/en'

def fetch_gbfs_feed(feed_name):
    """Busca um feed GBFS do Citi Bike."""
    url = f"{GBFS_BASE}/{feed_name}.json"
    logger.info(f"📡 Buscando GBFS feed: {feed_name}")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

# Station Information — dados cadastrais
try:
    station_info_raw = fetch_gbfs_feed('station_information')
    stations = station_info_raw['data']['stations']
    stations_df = pd.DataFrame(stations)
    
    # Selecionar colunas relevantes
    station_cols = [
        'station_id', 'name', 'lat', 'lon', 'capacity',
        'region_id', 'short_name', 'station_type',
    ]
    available_cols = [c for c in station_cols if c in stations_df.columns]
    stations_clean = stations_df[available_cols].copy()
    stations_clean['ingestion_timestamp'] = datetime.now()
    stations_clean['feed_timestamp'] = station_info_raw.get('last_updated', None)
    
    print(f" Total de estações: {len(stations_clean):,}")
    display(stations_clean.head())

except Exception as e:
    logger.error(f"❌ Erro ao buscar estações: {e}")
    stations_clean = pd.DataFrame()

22:12:32 [INFO] 📡 Buscando GBFS feed: station_information


 Total de estações: 2,360


,station_id,name,lat,lon,capacity,region_id,short_name,station_type,ingestion_timestamp,feed_timestamp
0,1905837242740508940,31 St & Broadway,40.762110,-73.925230,35,71,6789.20,classic,2026-04-07 22:12:34.275241,1775599944
1,41495491-5d89-4e14-aab9-c3db04aad399,43 St & Skillman Ave,40.746927,-73.920825,19,71,6325.01,classic,2026-04-07 22:12:34.275241,1775599944
2,2177014969129222184,Henry Hudson Pkwy E & W 231 St,40.883360,-73.915050,0,71,8838.06,classic,2026-04-07 22:12:34.275241,1775599944
3,2177016530523622712,Post Rd & W 251 St,40.895610,-73.897930,0,71,8998.03,classic,2026-04-07 22:12:34.275241,1775599944
4,2177016940364559212,David Sheridan Plaza & Broadway,40.905010,-73.896630,0,71,9074.06,classic,2026-04-07 22:12:34.275241,1775599944


In [16]:
# ============================================================
# 4.2 Station Status — snapshot de disponibilidade
# ============================================================

try:
    station_status_raw = fetch_gbfs_feed('station_status')
    status_list = station_status_raw['data']['stations']
    status_df = pd.DataFrame(status_list)
    
    status_cols = [
        'station_id', 'num_bikes_available', 'num_docks_available',
        'num_ebikes_available', 'is_installed', 'is_renting', 'is_returning',
        'last_reported',
    ]
    available_cols = [c for c in status_cols if c in status_df.columns]
    status_clean = status_df[available_cols].copy()
    status_clean['snapshot_timestamp'] = datetime.now()
    
    print(f"📊 Status coletado de {len(status_clean)} estações")
    print(f"   Bikes disponíveis no sistema: {status_clean['num_bikes_available'].sum():,}")
    if 'num_ebikes_available' in status_clean.columns:
        print(f"   E-bikes disponíveis: {status_clean['num_ebikes_available'].sum():,}")
    print(f"   Docks livres: {status_clean['num_docks_available'].sum():,}")

except Exception as e:
    logger.error(f"❌ Erro ao buscar status: {e}")
    status_clean = pd.DataFrame()

22:12:34 [INFO] 📡 Buscando GBFS feed: station_status


📊 Status coletado de 2360 estações
   Bikes disponíveis no sistema: 30,819
   E-bikes disponíveis: 12,996
   Docks livres: 34,548


In [17]:
# ============================================================
# 4.3 Gravar estações na Bronze
# ============================================================

if not stations_clean.empty:
    # Salvar como Parquet com PyArrow (evita dependência do Hadoop no Windows)
    _info_dir = BRONZE_STATIONS / 'station_info'
    _info_dir.mkdir(parents=True, exist_ok=True)
    _table = pa.Table.from_pandas(stations_clean)
    pq.write_table(_table, _info_dir / 'data.parquet', compression='snappy')
    del _table

    # Salvar também como JSON para o simulador de streaming
    stations_clean.to_json(
        str(BRONZE_STATIONS / 'station_info.json'),
        orient='records', indent=2
    )
    print(f" Station info gravado na Bronze (Parquet + JSON)")

if not status_clean.empty:
    _status_dir = BRONZE_STATIONS / 'station_status'
    _status_dir.mkdir(parents=True, exist_ok=True)
    _table = pa.Table.from_pandas(status_clean)
    pq.write_table(_table, _status_dir / 'data.parquet', compression='snappy')
    del _table
    print(f" Station status gravado na Bronze (Parquet)")


 Station info gravado na Bronze (Parquet + JSON)
 Station status gravado na Bronze (Parquet)


---
## 5. 📋 Resumo da Ingestão & Inventário da Bronze

In [18]:
# ============================================================
# 5.1 Inventário completo da camada Bronze
# ============================================================

def get_dir_size(path):
    """Calcula o tamanho total de um diretório."""
    total = 0
    for f in Path(path).rglob('*'):
        if f.is_file():
            total += f.stat().st_size
    return total

print("=" * 70)
print(" INVENTÁRIO DA CAMADA BRONZE (RAW LANDING)")
print("=" * 70)

inventory = []
for subdir in ['trips', 'weather', 'holidays', 'stations']:
    path = BRONZE_DIR / subdir
    if path.exists():
        size = get_dir_size(path)
        n_files = len(list(path.rglob('*.parquet')))
        inventory.append({
            'Dataset': subdir,
            'Arquivos Parquet': n_files,
            'Tamanho': f"{size / 1e6:.1f} MB" if size < 1e9 else f"{size / 1e9:.2f} GB",
            'Status': '✅' if n_files > 0 else '❌'
        })

inv_df = pd.DataFrame(inventory)
display(inv_df)

total_size = get_dir_size(BRONZE_DIR)
print(f"\n💾 Tamanho total da Bronze: {total_size / 1e9:.2f} GB")
print(f"📁 Localização: {BRONZE_DIR}")

 INVENTÁRIO DA CAMADA BRONZE (RAW LANDING)


,Dataset,Arquivos Parquet,Tamanho,Status
0,trips,289,4.51 GB,✅
1,weather,1,0.5 MB,✅
2,holidays,1,0.0 MB,✅
3,stations,2,1.0 MB,✅



💾 Tamanho total da Bronze: 4.51 GB
📁 Localização: /home/jovyan/work/dados/bronze


In [19]:
# ============================================================
# 5.2 Checklist de ingestão
# ============================================================

print("\n" + "=" * 70)
print("✅ CHECKLIST — ETAPA DE INGESTÃO (AV1)")
print("=" * 70)

checks = {
    'Citi Bike Trip Data (batch)': BRONZE_TRIPS.exists() and any(BRONZE_TRIPS.rglob('*.parquet')),
    'Weather Data (Open-Meteo)': BRONZE_WEATHER.exists() and any(BRONZE_WEATHER.rglob('*.parquet')),
    'Holidays (US + NY)': BRONZE_HOLIDAYS.exists() and any(BRONZE_HOLIDAYS.rglob('*.parquet')),
    'Station Info (GBFS JSON)': (BRONZE_STATIONS / 'station_info.json').exists(),
    'Station Status (GBFS)': (BRONZE_STATIONS / 'station_status').exists(),
    'Formato Parquet utilizado': True,
    'PySpark utilizado': True,
    'Particionamento por ano/mês': True,
}

for item, status in checks.items():
    icon = '✅' if status else '❌'
    print(f"  {icon} {item}")

print(f"\n{'='*70}")
completed = sum(checks.values())
total = len(checks)
print(f"  Progresso: {completed}/{total} ({100*completed/total:.0f}%)")


✅ CHECKLIST — ETAPA DE INGESTÃO (AV1)
  ✅ Citi Bike Trip Data (batch)
  ✅ Weather Data (Open-Meteo)
  ✅ Holidays (US + NY)
  ✅ Station Info (GBFS JSON)
  ✅ Station Status (GBFS)
  ✅ Formato Parquet utilizado
  ✅ PySpark utilizado
  ✅ Particionamento por ano/mês

  Progresso: 8/8 (100%)


In [20]:
# Encerrar SparkSession
spark.stop()
print(" SparkSession encerrada. Ingestão batch concluída!")

 SparkSession encerrada. Ingestão batch concluída!
